# STEP 1: Anthropic API 最初のリクエスト

このノートブックでは、Claude API への基本的なリクエストを実践します。

## 1. 必要なパッケージのインストール

In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. 環境変数の読み込みとクライアントの作成

In [3]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

print("クライアントの作成に成功しました")

クライアントの作成に成功しました


## 3. 最初のAPIリクエスト

In [4]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

print(message)

Message(id='msg_01V33crFU2vNWFPqv9Tzth5S', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computing that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain complex problems much faster than classical computers.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=40, server_tool_use=None, service_tier='standard'))


## 4. レスポンスからテキストを抽出

In [5]:
print(message.content[0].text)

Quantum computing is a type of computing that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain complex problems much faster than classical computers.


---

# STEP 2: 複数ターンの会話（会話履歴の管理）

Claude はリクエスト間で会話を記憶しません。
文脈を維持するには、**会話履歴をこちら側で管理**してリクエストに含める必要があります。

## 5. ヘルパー関数の定義

In [6]:
def add_user_message(messages, text):
    """ユーザーメッセージを履歴に追加する"""
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    """アシスタントメッセージを履歴に追加する"""
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    """会話履歴全体をClaudeに送信してレスポンスを返す"""
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

print("ヘルパー関数の定義が完了しました")

ヘルパー関数の定義が完了しました


## 6. 複数ターンの会話を実行する

In [7]:
# 空のメッセージリストからスタート
messages = []

# 最初の質問を追加
add_user_message(messages, "Define quantum computing in one sentence")

# Claudeの回答を取得
answer = chat(messages)
print(f"Claude (1回目): {answer}")

# Claudeの回答を履歴に追加
add_assistant_message(messages, answer)

# フォローアップの質問を追加
add_user_message(messages, "Write another sentence")

# 会話履歴全体を含めてリクエスト
final_answer = chat(messages)
print(f"Claude (2回目): {final_answer}")

Claude (1回目): Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.
Claude (2回目): Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are either 0 or 1.


## 7. 送信している会話履歴を確認する

In [8]:
# 2回目のリクエスト時点でのメッセージ履歴を確認
import json
print(json.dumps(messages, indent=2, ensure_ascii=False))

[
  {
    "role": "user",
    "content": "Define quantum computing in one sentence"
  },
  {
    "role": "assistant",
    "content": "Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers."
  },
  {
    "role": "user",
    "content": "Write another sentence"
  }
]


---

# STEP 3: システムプロンプト

システムプロンプトを使うと、Claude の振る舞い・トーン・役割をカスタマイズできます。
ユーザーの質問とは別に、会話全体を通じて有効な「指示」として機能します。

## 8. chat関数をシステムプロンプト対応に更新する

Claude の API は `system=None` を受け付けないため、指定がある場合だけ含める実装にします。

In [9]:
def chat(messages, system=None):
    """会話履歴全体をClaudeに送信してレスポンスを返す（システムプロンプト対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    # system=None の場合はパラメータに含めない
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 9. システムプロンプトなしで質問する（通常の回答）

In [10]:
messages = []
add_user_message(messages, "How do I solve: 5x + 2 = 3?")

answer = chat(messages)
print("【システムプロンプトなし】")
print(answer)

【システムプロンプトなし】
# Solving 5x + 2 = 3

To solve for x, isolate it by undoing the operations:

**Step 1:** Subtract 2 from both sides
```
5x + 2 - 2 = 3 - 2
5x = 1
```

**Step 2:** Divide both sides by 5
```
5x/5 = 1/5
x = 1/5
```

**Answer:** x = 1/5 (or 0.2 as a decimal)

**Check:** 5(1/5) + 2 = 1 + 2 = 3 ✓


## 10. システムプロンプトあり（数学家庭教師として振る舞う）

In [11]:
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

messages = []
add_user_message(messages, "How do I solve: 5x + 2 = 3?")

answer = chat(messages, system=system_prompt)
print("【システムプロンプトあり（数学家庭教師）】")
print(answer)

【システムプロンプトあり（数学家庭教師）】
Great question! Let's work through this together step by step.

First, let me ask you: **What do you think our goal is when solving an equation like this?**

Take a moment to think about what we're trying to find.


---

# STEP 4: Temperature（温度）

`temperature` は Claude の回答の「ランダム性・創造性」を制御するパラメータです。

| 温度 | 特性 | 用途例 |
|------|------|--------|
| 0.0〜0.3（低） | 決定論的・一貫性が高い | コード生成、データ抽出、事実回答 |
| 0.4〜0.7（中） | バランス型 | 要約、教育コンテンツ、問題解決 |
| 0.8〜1.0（高） | 多様・創造的 | ブレインストーミング、創作、マーケティング |

## 11. chat関数にtemperatureパラメータを追加する

In [12]:
def chat(messages, system=None, temperature=1.0):
    """会話履歴全体をClaudeに送信してレスポンスを返す（temperature対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 12. 低温（0.0）vs 高温（1.0）を比較する

同じプロンプトを温度違いで3回ずつ実行して、回答のばらつきを確認します。

In [13]:
prompt = "Give me a one-sentence movie idea about time travel."

print("=== 低温 (temperature=0.0) ===")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    answer = chat(messages, temperature=0.0)
    print(f"  {i+1}回目: {answer}")

print()
print("=== 高温 (temperature=1.0) ===")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    answer = chat(messages, temperature=1.0)
    print(f"  {i+1}回目: {answer}")

=== 低温 (temperature=0.0) ===
  1回目: A woman discovers she can send text messages to her past self, but each message she sends erases someone she loves from existence.
  2回目: A woman discovers she can send text messages to her past self, but each message she sends erases someone she loves from existence.
  3回目: A woman discovers she can send text messages to her past self, but each message she sends erases someone she loves from existence.

=== 高温 (temperature=1.0) ===
  1回目: A woman discovers she can send 10-second voice messages to her past self, but each message creates a new timeline where a different person she loves never existed.
  2回目: A time traveler discovers they can only jump to moments where they were genuinely happy, forcing them to confront why those moments have become so rare in their present life.
  3回目: A time traveler accidentally prevents their own birth and has only 24 hours to undo the damage before they fade from existence entirely.


---

# STEP 5: ストリーミング

通常のリクエストは応答が完成するまで待つのに対し、ストリーミングはテキストが生成されるたびにチャンクごとに受信できます。
チャットアプリのような「文字が流れてくる」UXを実現するための機能です。

## 13. ストリームイベントの中身を確認する

`stream=True` を指定すると、どんなイベントが送られてくるかを見てみます。

In [14]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01JPYBCWrvyWfKVYdnmbiS9u', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=2, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A fictional', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' customer relationship management system that tracks user interactions, purchase history, and sentiment', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(

## 14. テキストだけをリアルタイムで表示する

SDKの簡略インターフェース `messages.stream` を使うと、テキスト以外のイベントを自動でフィルタリングできます。
`end=""` により改行なしで文字が流れるように表示されます。

In [15]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

A fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but contains synthetic, randomly generated, or placeholder data used for testing, development, or demonstration purposes without connecting to actual production data.

## 15. ストリーミングしながら完全なメッセージも取得する

リアルタイム表示と同時に、DB保存や後続処理用の完全なメッセージオブジェクトも取得できます。

In [16]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    # リアルタイムでテキストを表示
    for text in stream.text_stream:
        print(text, end="", flush=True)

    # ストリーミング完了後に完全なメッセージを取得
    final_message = stream.get_final_message()

print("\n\n--- final_message ---")
print(f"stop_reason : {final_message.stop_reason}")
print(f"input_tokens: {final_message.usage.input_tokens}")
print(f"output_tokens: {final_message.usage.output_tokens}")

A fake database is a simulated or mock data storage system used for testing, development, or demonstration purposes that mimics the structure and behavior of a real database without containing actual production data.

--- final_message ---
stop_reason : end_turn
input_tokens: 18
output_tokens: 41


---

# STEP 6: 構造化データの出力制御（プリフィル + 停止シーケンス）

ClaudeはデフォルトでJSONをマークダウンコードブロックや説明文で囲んで返します。
**アシスタントメッセージのプリフィル**と**停止シーケンス**を組み合わせることで、生データだけを取得できます。

## 16. デフォルト（プリフィルなし）の出力を確認する

In [17]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
print(text)

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}
```


## 17. chat関数に stop_sequences パラメータを追加する

In [18]:
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """会話履歴全体をClaudeに送信してレスポンスを返す（stop_sequences対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 18. プリフィル + 停止シーケンスで生JSONだけを取得する

- `add_assistant_message(messages, "```json")` でClaudeに「コードブロックはもう始まっている」と思わせる
- `stop_sequences=["` ``` `"]` でClaudeが閉じタグを書こうとした瞬間に停止させる

In [19]:
import json

messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")  # プリフィル：コードブロックが既に始まっているように見せる

text = chat(messages, stop_sequences=["```"])  # 閉じタグが来たら停止

print("--- 生のレスポンス ---")
print(repr(text))

print("\n--- パース済みJSON ---")
clean_json = json.loads(text.strip())
print(json.dumps(clean_json, indent=2))

--- 生のレスポンス ---
'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

--- パース済みJSON ---
{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "running"
    ]
  }
}


---

# STEP 7: プロンプト評価ワークフロー

プロンプトを「なんとなくテスト」するのではなく、スコアで客観的に測定・比較する仕組みを作ります。

**5ステップの流れ：**
1. プロンプトを作成する
2. 評価データセットを用意する
3. 各質問をClaudeに送って回答を得る
4. 採点者（Claude）にスコアをつけさせる
5. プロンプトを改善して繰り返す

## 19. STEP1〜3: プロンプトと評価データセットを用意し、Claudeに回答させる

In [20]:
# STEP1: 評価対象のプロンプト（シンプルな初期バージョン）
def build_prompt(question):
    return f"""Please answer the user's question:

{question}"""

# STEP2: 評価データセット
questions = [
    "What's 2+2?",
    "How do you make oatmeal?",
    "How far away is the moon?",
]

# STEP3: 各質問をClaudeに送って回答を収集
answers = []
for question in questions:
    messages = []
    add_user_message(messages, build_prompt(question))
    answer = chat(messages, temperature=0.0)  # 再現性のため低温に設定
    answers.append(answer)
    print(f"Q: {question}")
    print(f"A: {answer[:100]}...")  # 長い場合は先頭100文字だけ表示
    print()

Q: What's 2+2?
A: 2 + 2 = 4...

Q: How do you make oatmeal?
A: # How to Make Oatmeal

## Basic Stovetop Method

**Ingredients:**
- 1 cup rolled oats (old-fashioned...

Q: How far away is the moon?
A: The Moon is approximately **238,855 miles** (384,400 kilometers) away from Earth on average.

Howeve...



## 20. STEP4: 採点者（Claude）にスコアをつけさせる

In [22]:
def grade_answer(question, answer):
    """Claudeに回答を1〜10で採点させる"""
    grader_prompt = f"""You are an objective grader. Score the following answer on a scale of 1-10.
10 = perfect answer, 1 = completely wrong or unhelpful.
Reply with only a single integer between 1 and 10. Do not include any other text.

Question: {question}
Answer: {answer}"""

    messages = []
    add_user_message(messages, grader_prompt)
    score_text = chat(messages, temperature=0.0)
    return int(score_text.strip())

# 各回答をスコアリング
scores = []
for question, answer in zip(questions, answers):
    score = grade_answer(question, answer)
    scores.append(score)
    print(f"Q: {question}")
    print(f"スコア: {score}/10")
    print()

avg_score = sum(scores) / len(scores)
print(f"平均スコア: {avg_score:.2f}/10")

Q: What's 2+2?
スコア: 10/10

Q: How do you make oatmeal?
スコア: 10/10

Q: How far away is the moon?
スコア: 10/10

平均スコア: 10.00/10


## 21. STEP5: プロンプトを改善して再評価する

プロンプトに「詳しく答えてください」を追加して、スコアが変わるか比較します。

In [23]:
# 改善版プロンプト
def build_prompt_v2(question):
    return f"""Please answer the user's question:

{question}

Answer the question with ample detail."""

# 改善版で回答を収集してスコアリング
scores_v2 = []
for question in questions:
    messages = []
    add_user_message(messages, build_prompt_v2(question))
    answer = chat(messages, temperature=0.0)
    score = grade_answer(question, answer)
    scores_v2.append(score)
    print(f"Q: {question}")
    print(f"スコア: {score}/10")
    print()

avg_score_v2 = sum(scores_v2) / len(scores_v2)

print(f"初期プロンプト 平均スコア: {avg_score:.2f}/10")
print(f"改善プロンプト 平均スコア: {avg_score_v2:.2f}/10")
print(f"差分: {avg_score_v2 - avg_score:+.2f}")

Q: What's 2+2?
スコア: 10/10

Q: How do you make oatmeal?
スコア: 9/10

Q: How far away is the moon?
スコア: 9/10

初期プロンプト 平均スコア: 10.00/10
改善プロンプト 平均スコア: 9.33/10
差分: -0.67
